In [ ]:
import json
import os
from urllib.parse import quote as url_quote

from requests import delete, get, post

# Generic utils
from utils import setup_notebook

# Check environment setup
if not setup_notebook(
    required_keys=[
        "AZURE_API_KEY",
        "AZURE_ENDPOINT_URL",
        "AZURE_DEPLOYMENT_NAME",
        "AZURE_DEPLOYMENT_NAME_EMBEDDINGS",
        "AZURE_API_VERSION",
    ],
):
    raise OSError("Please set up required environment variables")

In [ ]:
NAME = "Test Agent Azure: examples/create-agent-azure.ipynb"

response = post(
    "http://localhost:8000/api/v2/agents",
    json={
        "mode": "conversational",
        "name": NAME,
        "version": "1.0.0",
        "description": ("This is a test agent created by the examples/create-agent.ipynb script."),
        "runbook": "# Objective\nYou are a helpful assistant.",
        "platform_configs": [
            {
                "kind": "azure",
                "azure_api_key": os.environ["AZURE_API_KEY"],
                "azure_endpoint_url": os.environ["AZURE_ENDPOINT_URL"],
                "azure_deployment_name": os.environ["AZURE_DEPLOYMENT_NAME"],
                "azure_deployment_name_embeddings": os.environ["AZURE_DEPLOYMENT_NAME_EMBEDDINGS"],
                "azure_api_version": os.environ["AZURE_API_VERSION"],
            },
        ],
        "action_packages": [],
        "mcp_servers": [
            {
                "name": "agent-server",
                "url": "http://localhost:8000/api/v2/public-mcp",
            },
        ],
        "agent_architecture": {
            "name": "agent_platform.architectures.default",
            "version": "1.0.0",
        },
        "observability_configs": [],
        "question_groups": [],
        "extra": {
            "test": "test",
        },
    },
)
print(response.json())

In [3]:
name_url_encoded = url_quote(NAME)
agent_by_name = get(
    f"http://localhost:8000/api/v2/agents/by-name?name={name_url_encoded}",
).json()

# Get id to use in the next few examples
agent_id = agent_by_name["agent_id"]

In [ ]:
from widget import DebugChatWidget

widget = DebugChatWidget(
    agent_id=agent_id,
    base_url="http://localhost:8000/api/v2",
)
widget

In [ ]:
# Now list threads for the agent
specific_thread = get(
    f"http://localhost:8000/api/v2/threads?agent_id={agent_id}",
).json()
print(json.dumps(specific_thread, indent=2))

In [ ]:
specific_agent = get(f"http://localhost:8000/api/v2/agents/{agent_id}").json()
print(json.dumps(specific_agent, indent=2))

In [ ]:
delete(f"http://localhost:8000/api/v2/agents/{agent_id}")

In [ ]:
# If we get the agent again, it should be gone
specific_agent = get(f"http://localhost:8000/api/v2/agents/{agent_id}")
print(
    json.dumps(
        {
            "status_code": specific_agent.status_code,
            "response": specific_agent.json(),
        },
        indent=2,
    ),
)

In [ ]:
sync_conversation = post(
    f"http://localhost:8000/api/v2/runs/{agent_id}/sync",
    json={
        "agent_id": agent_id,
        "name": "Sync Conversation 1",
        "messages": [
            {
                "role": "user",
                "content": [{"kind": "text", "text": "What tools do you have?"}],
            }
        ],
    },
)

print(sync_conversation.json())